# 01 EDA — Predicting Smartphone Addiction

Official data facts only. No model training in this notebook.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..') if Path('data/raw').exists() is False and Path('../data/raw').exists() else Path('.')
RAW = ROOT / 'data' / 'raw'
FIG = ROOT / 'reports' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / 'train.csv')
test = pd.read_csv(RAW / 'test.csv')
sample = pd.read_csv(RAW / 'sample_submission.csv')
print({'train': train.shape, 'test': test.shape, 'sample': sample.shape})
print('target rate', train['addicted_label'].mean())
print(train.isna().mean().sort_values(ascending=False).head(12))


In [ ]:
# Target and screen-time distributions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train['addicted_label'].value_counts(normalize=True).plot(kind='bar', ax=axes[0], title='Target share')
train['daily_screen_time_hours'].hist(bins=40, ax=axes[1])
axes[1].set_title('daily_screen_time_hours')
fig.tight_layout()
fig.savefig(FIG / 'eda_target_screen.png', dpi=120)
plt.close(fig)


In [ ]:
# Train vs test numeric means for shared columns
shared = [c for c in train.columns if c in test.columns and c != 'id']
num = [c for c in shared if pd.api.types.is_numeric_dtype(train[c])]
cmp = pd.DataFrame({'train_mean': train[num].mean(), 'test_mean': test[num].mean()})
cmp['abs_diff'] = (cmp['train_mean'] - cmp['test_mean']).abs()
display(cmp.sort_values('abs_diff', ascending=False).head(12))
cmp.to_csv(FIG / 'eda_train_test_numeric_means.csv')
